# Lesson 01 - Johdanto tekoälyagentteihin

Tervetuloa **AI Agents for Beginners** -kurssin ensimmäiseen oppituntiin!

**Tekoälyagentti** on ohjelma, joka käyttää suurta kielimallia (LLM) päättelymoottorinaan ja voi suorittaa *toimintoja* oikeassa maailmassa — kutsua rajapintoja, tehdä tietokantakyselyjä tai ajaa koodia — saavuttaakseen käyttäjän puolesta asetetun tavoitteen.

Tässä muistikirjassa rakennat ensimmäisen agenttisi: **Matka-agentin**, joka suosittelee lomakohteita. Matkan varrella opit kuinka:

1. Yhdistää Azure AI Foundry Agent Serviceen käyttäen **Microsoft Agent Frameworkia**.
2. Antaa agentille **työkalu** — tavallisen Python-funktion, jota se voi kutsua.
3. Ajaa agentin ja tarkastella sen vastausta.
4. Suoratoistaa agentin vastaus token tokenilta.


## Asennus

Ennen kuin suoritat tämän muistikirjan, varmista, että sinulla on:

1. **Azure AI Foundry -projekti**, jossa on käytössä chat-malli (esim. `gpt-4o-mini`).
2. **Kirjautunut sisään Azure CLI:llä** — suorita `az login` komentorivilläsi.
3. **Asetettu tarvittavat ympäristömuuttujat:**
   - `AZURE_AI_PROJECT_ENDPOINT` — Azure AI Foundry -projektisi päätepiste.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — käytössä olevan mallin nimi.

Alla oleva solu asentaa tarvitsemasi Python-kirjastot.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Ensimmäisen agenttisi luominen

Agentti tarvitsee kaksi asiaa:

- **Ohjeet**, jotka kertovat sille *kuka se on* ja *miten sen tulee käyttäytyä* (järjestelmäkehotus).
- **Työkalut** — Python-funktiot, joissa on `@tool`-koristaja, joita agentti voi kutsua saadakseen tietoa tai suorittaakseen toimintoja.

Alla määrittelemme yksinkertaisen työkalun, joka palauttaa listan suosituista lomakohteista. Agentti käyttää tätä työkalua, kun käyttäjä pyytää matkasuosituksia.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Suoratoistovastaukset

Vuorovaikutteisemman kokemuksen saamiseksi voit **suoratoistaa** agentin vastauksen. Sen sijaan, että odottaisit täyttä vastausta, agentti tuottaa tekstinpätkiä sitä mukaa kun ne syntyvät. Tämä on erityisen hyödyllistä keskustelukäyttöliittymissä, joissa haluat näyttää tuloksen reaaliajassa.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Yhteenveto

Tässä oppitunnissa opit:

- **Luomaan tarjoajan**, joka muodostaa yhteyden Azure AI Foundry Agent Serviceen `AzureAIProjectAgentProvider`-luokan avulla.
- **Määrittelemään työkalun** käyttämällä `@tool`-koristetta, jotta agentti voi kutsua Python-funktioitasi.
- **Suorittamaan agentin** käyttäjän viestillä ja tulostamaan sen vastauksen.
- **Suoratoistamaan vastaukset** reaaliaikaista tulostusta varten.

Seuraavassa oppitunnissa tutustumme agenttipohjaisiin kehyksiin syvällisemmin ja opimme antamaan agenteille tehokkaampia työkaluja ja monivaiheista päättelykykyä.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
